# NB13 — Visualización Interactiva Completa (V1-V8)

**Objetivo:** Explorar TODAS las alternativas de visualización con PyVis interactivo.

**V1-V8:**
- V1: Árbol argumentación Fierro (Q→Hip→Evid→Resp)
- V2: Trees de predicados lógicos (OR/AND)
- V3: Cadena certificación (doc→claim→cert→status)
- V4: Timeline supersedes (evolución claims)
- V5: Comunidades grafo completo (71 clusters)
- V6: Heatmaps expandidos (source×kind, source×status)
- V7: Bridge tasks↔grafo (bipartite)
- V8: Capas grafo multi-layer

**PyVis:** HTML interactivo — zoom, pan, hover con labels, click para expandir.

Plan: `docs/plans/nb13_interactive_full.vsm`.

In [1]:
import sys, json, os, re
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

# Set VOSLAB_PATH so load.py finds data without vos_kernel
os.environ.setdefault("VOSLAB_PATH", str(Path.cwd()))

sys.path.insert(0, "packages/framework")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import networkx as nx
from pyvis.network import Network

from epistemic_graph import build, by_kind, by_edge, communities

print(f"imports OK — pyvis, matplotlib {matplotlib.__version__}, networkx {nx.__version__}")
print(f"VOSLAB_PATH={os.environ['VOSLAB_PATH']}")

imports OK — pyvis, matplotlib 3.10.7+dfsg1, networkx 3.4.2


In [2]:
# CELL 1: V1 — Árbol argumentación Fierro (Q→Hip→Evid→Resp) interactivo
# Estructura: 3 objections → hypothesis → evidence → response proposition
fierro_tree = {
    "Q1": {
        "topic": "Convexidad de Ω",
        "objection": "Ω no es convexo → Brouwer no aplica",
        "hypothesis": "¿Existe punto fijo sin convexidad?",
        "evidence": ["Krylov-Bogolyubov (Feller + compacto)", "Banach (contracción en Ω_A)", "λ_max = ln(0.99) < 0"],
        "response": "Prop1: Existencia SIN convexidad",
        "status": "RESUELTA",
    },
    "Q2": {
        "topic": "Invariancia de f₀",
        "objection": "Proyección deforma la dinámica",
        "hypothesis": "¿f₀(Ω) ⊆ Ω sin imponer?",
        "evidence": ["L_f ∈ [1.4, 24.4] (código desplegado)", "Zero exits N=3000/level", "Recovery time medido"],
        "response": "Confinamiento medido, no impuesto",
        "status": "RESUELTA",
    },
    "Q3": {
        "topic": "Ruido no acotado",
        "objection": "Invariancia puntual = falso",
        "hypothesis": "¿Viabilidad estocástica?",
        "evidence": ["Foster-Lyapunov drift", "Meyn-Tweedie ergodicity", "Empirical exit rate < 3/N"],
        "response": "Prop2: Viabilidad estocástica",
        "status": "RESUELTA",
    },
}

# Build PyVis network
net = Network(height="700px", width="100%", directed=True, bgcolor="#1a1a2e", font_color="white")
net.barnes_hut(gravity=-8000, central_gravity=0.3, spring_length=150)

# Root node
net.add_node("ROOT", label="FIerro\nObjections", color="#FF6B6B", size=40, font={"size": 18, "bold": True})

status_colors = {"RESUELTA": "#7ED321", "ABIERTA": "#F5A623"}
for q, d in fierro_tree.items():
    # Q node
    net.add_node(q, label=f"{q}\n{d['topic']}", color="#4A90D9", size=30, font={"size": 14, "bold": True})
    net.add_edge("ROOT", q, color="#888888", arrows="to")
    
    # Hypothesis
    hyp_id = f"{q}_HYP"
    net.add_node(hyp_id, label=d["hypothesis"], color="#BD10E0", shape="box", size=25, font={"size": 11})
    net.add_edge(q, hyp_id, color="#BD10E0", arrows="to")
    
    # Evidence nodes
    for i, ev in enumerate(d["evidence"]):
        ev_id = f"{q}_EV{i}"
        net.add_node(ev_id, label=ev, color="#50E3C2", shape="ellipse", size=20, font={"size": 10})
        net.add_edge(hyp_id, ev_id, color="#50E3C2", arrows="to")
    
    # Response
    resp_id = f"{q}_RESP"
    net.add_node(resp_id, label=d["response"], color=status_colors[d["status"]], shape="diamond", size=30, font={"size": 12, "bold": True})
    net.add_edge(hyp_id, resp_id, color=status_colors[d["status"]], arrows="to", width=3)

# Save interactive HTML
net.save_graph("/tmp/nb13_v1_fierro_tree.html")
net.show("/tmp/nb13_v1_fierro_tree.html", notebook=False)
print("V1: Fierro argumentation tree — /tmp/nb13_v1_fierro_tree.html")
print(f"  3 objections, 3 hypotheses, 9 evidence nodes, 3 responses (all RESUELTA)")

/tmp/nb13_v1_fierro_tree.html
V1: Fierro argumentation tree — /tmp/nb13_v1_fierro_tree.html
  3 objections, 3 hypotheses, 9 evidence nodes, 3 responses (all RESUELTA)


In [3]:
# CELL 2: V2 — Trees de predicados lógicos (OR/AND parseados)
# Build graph first
g = build()
print(f"Graph built: {g['meta']['nodes_total']} nodes / {g['meta']['edges_total']} edges")

# Extract predicates from graph and parse their OR/AND structure
pred_nodes = [n for n in g["nodes"] if n["kind"] == "predicate"]
print(f"Total predicates: {len(pred_nodes)}")

def parse_predicate_tree(pred_text):
    """Parse OR(AND(...), ...) into tree structure."""
    # Simple recursive descent parser for OR/AND
    tokens = re.findall(r'OR|AND|NOT|\(|\)|[^\s(),]+', pred_text)
    pos = [0]
    def parse():
        if pos[0] >= len(tokens):
            return {"type": "empty"}
        tok = tokens[pos[0]]
        if tok in ("OR", "AND"):
            pos[0] += 1
            children = []
            if pos[0] < len(tokens) and tokens[pos[0]] == "(":
                pos[0] += 1
                while pos[0] < len(tokens) and tokens[pos[0]] != ")":
                    children.append(parse())
                    if pos[0] < len(tokens) and tokens[pos[0]] == ",":
                        pos[0] += 1
                if pos[0] < len(tokens):
                    pos[0] += 1  # skip )
            return {"type": tok, "children": children}
        elif tok == "NOT":
            pos[0] += 1
            child = parse()
            return {"type": "NOT", "children": [child]}
        elif tok == "(":
            pos[0] += 1
            child = parse()
            if pos[0] < len(tokens) and tokens[pos[0]] == ")":
                pos[0] += 1
            return child
        else:
            pos[0] += 1
            return {"type": "leaf", "value": tok[:40]}
    return parse()

# Visualize top 3 predicates as trees
net2 = Network(height="600px", width="100%", directed=True, bgcolor="#1a1a2e", font_color="white")
net2.barnes_hut(gravity=-5000, central_gravity=0.3)

# Pick 3 predicates with OR structure (most complex)
or_preds = [n for n in pred_nodes if "OR" in n.get("label", "")][:3]
node_counter = [0]

def add_tree_to_network(net, tree, parent_id, depth=0):
    node_counter[0] += 1
    nid = f"p{node_counter[0]}"
    if tree["type"] == "leaf":
        net.add_node(nid, label=tree.get("value", ""), color="#50E3C2", shape="box", size=15, font={"size": 9})
    elif tree["type"] == "OR":
        net.add_node(nid, label="OR", color="#FF6B6B", shape="circle", size=25, font={"size": 12, "bold": True})
    elif tree["type"] == "AND":
        net.add_node(nid, label="AND", color="#4A90D9", shape="circle", size=25, font={"size": 12, "bold": True})
    elif tree["type"] == "NOT":
        net.add_node(nid, label="NOT", color="#F5A623", shape="circle", size=20, font={"size": 11, "bold": True})
    else:
        net.add_node(nid, label=tree["type"], color="#CCCCCC", size=15)
    
    if parent_id:
        net.add_edge(parent_id, nid, color="#888888", arrows="to")
    
    for child in tree.get("children", []):
        add_tree_to_network(net, child, nid, depth+1)

for i, pred in enumerate(or_preds):
    root_id = f"pred_root_{i}"
    net2.add_node(root_id, label=f"Pred {i+1}\n{pred['id'][:30]}", color="#BD10E0", size=30, font={"size": 11, "bold": True})
    tree = parse_predicate_tree(pred.get("label", ""))
    add_tree_to_network(net2, tree, root_id)

net2.save_graph("/tmp/nb13_v2_predicate_trees.html")
net2.show("/tmp/nb13_v2_predicate_trees.html", notebook=False)
print(f"V2: Predicate trees — /tmp/nb13_v2_predicate_trees.html")
print(f"  {len(or_preds)} predicates visualized as OR/AND trees")

Graph built: 2227 nodes / 4962 edges
Total predicates: 45
/tmp/nb13_v2_predicate_trees.html
V2: Predicate trees — /tmp/nb13_v2_predicate_trees.html
  3 predicates visualized as OR/AND trees


In [4]:
# CELL 3: V3 — Cadena certificación (doc→claim→cert→status) DAG
# Build certification chain from graph edges
cert_edges = [e for e in g["edges"] if e["kind"] in ("extracted_from", "certified_by")]
print(f"Certification edges: {len(cert_edges)}")

# Get nodes involved in certification
cert_nodes_ids = set()
for e in cert_edges:
    cert_nodes_ids.add(e["source"])
    cert_nodes_ids.add(e["target"])

# Subgraph for certification chain (limit to 150 for readability)
cert_sub = [n for n in g["nodes"] if n["id"] in cert_nodes_ids][:150]
cert_sub_ids = {n["id"] for n in cert_sub}
print(f"Certification subgraph: {len(cert_sub)} nodes")

# Filter edges to only those where BOTH endpoints are in subgraph
cert_edges_filtered = [e for e in cert_edges if e["source"] in cert_sub_ids and e["target"] in cert_sub_ids]
print(f"Filtered edges: {len(cert_edges_filtered)}")

# Build PyVis network
net3 = Network(height="700px", width="100%", directed=True, bgcolor="#1a1a2e", font_color="white")
net3.barnes_hut(gravity=-8000, central_gravity=0.2, spring_length=200)

kind_colors = {
    "claim": "#4A90D9", "answer": "#7ED321", "certificate": "#50E3C2",
    "document": "#8B572A", "rule": "#D0021B", "experiment": "#9013FE",
}
status_colors = {"DECLARED": "#F5A623", "OBSERVED": "#4A90D9", "CERTIFIED": "#7ED321", "REFUTED": "#D0021B"}

# Add ALL nodes first
for n in cert_sub:
    kind = n.get("kind", "unknown")
    status = n.get("dims", {}).get("status") or n.get("status", "")
    color = status_colors.get(status, kind_colors.get(kind, "#CCCCCC"))
    label = f"{kind[:4]}:{n['id'][:25]}"
    net3.add_node(n["id"], label=label, color=color, size=20, font={"size": 9},
                  title=f"Kind: {kind}\nStatus: {status}")

# Verify all nodes added
added_ids = set(net3.get_nodes())
print(f"Nodes added to net3: {len(added_ids)}")
missing = cert_sub_ids - added_ids
if missing:
    print(f"WARNING: {len(missing)} nodes not added: {list(missing)[:3]}")

# Add edges (now safe - all nodes exist)
for e in cert_edges_filtered:
    edge_color = "#50E3C2" if e["kind"] == "certified_by" else "#8B572A"
    net3.add_edge(e["source"], e["target"], color=edge_color, arrows="to")

net3.save_graph("/tmp/nb13_v3_cert_chain.html")
net3.show("/tmp/nb13_v3_cert_chain.html", notebook=False)
print(f"V3: Certification chain — /tmp/nb13_v3_cert_chain.html")
print(f"  {len(cert_sub)} nodes, {len(cert_edges_filtered)} edges (extracted_from + certified_by)")

Certification edges: 1836
Certification subgraph: 150 nodes
Filtered edges: 19
Nodes added to net3: 150
/tmp/nb13_v3_cert_chain.html
V3: Certification chain — /tmp/nb13_v3_cert_chain.html
  150 nodes, 19 edges (extracted_from + certified_by)


In [5]:
# CELL 4: V4 — Timeline supersedes (evolución claims)
# supersedes edges show claim evolution
supersedes_edges = [e for e in g["edges"] if e["kind"] == "supersedes"]
print(f"Supersedes edges: {len(supersedes_edges)}")

# Build evolution chains
net4 = Network(height="600px", width="100%", directed=True, bgcolor="#1a1a2e", font_color="white")
net4.barnes_hut(gravity=-3000, central_gravity=0.3, spring_length=120)

# Get nodes involved in supersedes
sup_nodes_ids = set()
for e in supersedes_edges:
    sup_nodes_ids.add(e["source"])
    sup_nodes_ids.add(e["target"])

# Color gradient by depth in chain (old=red, new=green)
# Find root claims (not superseded by anything)
superseded_ids = {e["target"] for e in supersedes_edges}
root_ids = sup_nodes_ids - superseded_ids

# BFS to assign depth
depth = {}
for root in root_ids:
    depth[root] = 0
    queue = [root]
    while queue:
        curr = queue.pop(0)
        for e in supersedes_edges:
            if e["source"] == curr:
                child = e["target"]
                depth[child] = depth[curr] + 1
                queue.append(child)

# Add nodes with color gradient
max_depth = max(depth.values()) if depth else 1
for nid in sup_nodes_ids:
    d = depth.get(nid, 0)
    # Gradient: red (old) -> yellow -> green (new)
    ratio = d / max_depth if max_depth > 0 else 0
    r = int(255 * (1 - ratio))
    g_val = int(255 * ratio)
    color = f"#{r:02x}{g_val:02x}00"
    node = next((n for n in g["nodes"] if n["id"] == nid), None)
    label = f"d{d}:{nid[:25]}" if node else f"d{d}:{nid[:25]}"
    net4.add_node(nid, label=label, color=color, size=20, font={"size": 9})

# Add edges
for e in supersedes_edges:
    src_depth = depth.get(e["source"], 0)
    tgt_depth = depth.get(e["target"], 0)
    net4.add_edge(e["source"], e["target"], color="#888888", arrows="to",
                  title=f"depth {src_depth} to {tgt_depth}")

net4.save_graph("/tmp/nb13_v4_supersedes_timeline.html")
net4.show("/tmp/nb13_v4_supersedes_timeline.html", notebook=False)
print(f"V4: Supersedes timeline — /tmp/nb13_v4_supersedes_timeline.html")
print(f"  {len(sup_nodes_ids)} nodes, {len(supersedes_edges)} edges")
print(f"  Chain depth: 0 to {max_depth} (red=old, green=new)")

Supersedes edges: 240
/tmp/nb13_v4_supersedes_timeline.html
V4: Supersedes timeline — /tmp/nb13_v4_supersedes_timeline.html
  318 nodes, 240 edges
  Chain depth: 0 to 1 (red=old, green=new)


In [6]:
# CELL 5: V5 — Comunidades grafo completo (clusters, labels)
comm = communities(g, ("contradicts", "supersedes"))
comm_sizes = Counter(comm.values())
print(f"Total communities: {len(comm_sizes)}")
print(f"Top 10 sizes: {[s for _,s in comm_sizes.most_common(10)]}")

# Build PyVis network with community coloring
net5 = Network(height="800px", width="100%", directed=False, bgcolor="#1a1a2e", font_color="white")
net5.barnes_hut(gravity=-10000, central_gravity=0.1, spring_length=300)

# Community colors (cycle through distinct colors)
comm_colors = ["#FF6B6B", "#4A90D9", "#7ED321", "#F5A623", "#BD10E0", "#50E3C2", "#D0021B", "#9013FE", "#FF9500", "#00BCD4"]

# Group nodes by community
comm_nodes = defaultdict(list)
for nid, cid in comm.items():
    comm_nodes[cid].append(nid)

# Add top 15 communities (largest) for readability — use index for color (cid is string)
top_comms = [cid for cid, _ in comm_sizes.most_common(15)]
for idx, cid in enumerate(top_comms):
    color = comm_colors[idx % len(comm_colors)]
    nodes_in_comm = comm_nodes[cid][:20]
    for nid in nodes_in_comm:
        node = next((n for n in g["nodes"] if n["id"] == nid), None)
        kind = node.get("kind", "?") if node else "?"
        label = f"{kind[:4]}:{nid[:20]}"
        title = f"Community {idx+1}\nKind: {kind}"
        net5.add_node(nid, label=label, color=color, size=15, font={"size": 8}, title=title)

# Add edges within communities
for e in g["edges"][:2000]:
    if e["source"] in comm and e["target"] in comm:
        if comm[e["source"]] == comm[e["target"]]:
            src_comm = comm[e["source"]]
            if src_comm in top_comms:
                net5.add_edge(e["source"], e["target"], color="#888888", width=0.5)

net5.save_graph("/tmp/nb13_v5_communities.html")
net5.show("/tmp/nb13_v5_communities.html", notebook=False)
print(f"V5: Communities — /tmp/nb13_v5_communities.html")
print(f"  Top 15 communities shown (20 nodes each max)")

Traceback (most recent call last):
  File "/tmp/opencode/run_notebook.py", line 27, in <module>
    exec(compile(src, f"<cell {i}>", "exec"), ns)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<cell 6>", line 22, in <module>
TypeError: not all arguments converted during string formatting


Exception: not all arguments converted during string formatting

In [ ]:
# CELL 6: V6 — Heatmaps expandidos (source×kind, source×status, kind×cert)
# Parse sidecars for source→kind→status→risk distribution
sidecars_dir = Path("state/epistemic/sidecars")
source_data = defaultdict(lambda: {"kind": Counter(), "status": Counter(), "risk": Counter(), "method": Counter()})

for vf in sidecars_dir.glob("*.claims.vsm"):
    text = vf.read_text()
    # Extract multiple fields per claim
    for m in re.finditer(r'"source_id":\s*"([^"]+)".*?"type":\s*"([^"]+)".*?"status":\s*"([^"]+)".*?"risk":\s*"([^"]+)"', text, re.DOTALL):
        sid, typ, status, risk = m.groups()
        source_data[sid]["kind"][typ] += 1
        source_data[sid]["status"][status] += 1
        source_data[sid]["risk"][risk] += 1

# Take top 15 sources
top_sources = sorted(source_data.keys(), key=lambda s: sum(source_data[s]["kind"].values()), reverse=True)[:15]

# 3 heatmaps side by side
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
fig.suptitle("Heatmaps Expandidos — Source × (Kind, Status, Risk)", fontsize=14, fontweight="bold")

# Heatmap 1: Source × Kind
all_kinds = sorted({k for s in top_sources for k in source_data[s]["kind"]})
mat1 = np.zeros((len(top_sources), len(all_kinds)), dtype=int)
for i, sid in enumerate(top_sources):
    for j, kind in enumerate(all_kinds):
        mat1[i,j] = source_data[sid]["kind"].get(kind, 0)
im1 = axes[0].imshow(mat1, cmap="YlOrRd", aspect="auto")
axes[0].set_xticks(range(len(all_kinds)))
axes[0].set_xticklabels(all_kinds, rotation=45, ha="right")
axes[0].set_yticks(range(len(top_sources)))
axes[0].set_yticklabels([s[:25] for s in top_sources], fontsize=8)
axes[0].set_title("Source × Kind")
plt.colorbar(im1, ax=axes[0])

# Heatmap 2: Source × Status
all_statuses = sorted({k for s in top_sources for k in source_data[s]["status"]})
mat2 = np.zeros((len(top_sources), len(all_statuses)), dtype=int)
for i, sid in enumerate(top_sources):
    for j, status in enumerate(all_statuses):
        mat2[i,j] = source_data[sid]["status"].get(status, 0)
im2 = axes[1].imshow(mat2, cmap="Blues", aspect="auto")
axes[1].set_xticks(range(len(all_statuses)))
axes[1].set_xticklabels(all_statuses, rotation=45, ha="right")
axes[1].set_yticks(range(len(top_sources)))
axes[1].set_yticklabels([s[:25] for s in top_sources], fontsize=8)
axes[1].set_title("Source × Status")
plt.colorbar(im2, ax=axes[1])

# Heatmap 3: Source × Risk
all_risks = sorted({k for s in top_sources for k in source_data[s]["risk"]})
mat3 = np.zeros((len(top_sources), len(all_risks)), dtype=int)
for i, sid in enumerate(top_sources):
    for j, risk in enumerate(all_risks):
        mat3[i,j] = source_data[sid]["risk"].get(risk, 0)
im3 = axes[2].imshow(mat3, cmap="Oranges", aspect="auto")
axes[2].set_xticks(range(len(all_risks)))
axes[2].set_xticklabels(all_risks, rotation=45, ha="right")
axes[2].set_yticks(range(len(top_sources)))
axes[2].set_yticklabels([s[:25] for s in top_sources], fontsize=8)
axes[2].set_title("Source × Risk")
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.savefig("/tmp/nb13_v6_heatmaps.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"V6: 3 heatmaps — /tmp/nb13_v6_heatmaps.png")
print(f"  {len(top_sources)} sources × {len(all_kinds)} kinds / {len(all_statuses)} statuses / {len(all_risks)} risks")

In [ ]:
# CELL 7: V7 — Bridge tasks↔grafo (bipartite, connectivity)
from epistemic_graph import bridge_view
v = bridge_view(g)
print(f"Bridge view: {v['denominators']['tasks_live']} live tasks, {v['denominators']['tasks_matched']} matched")

# Build bipartite network: tasks (left) ↔ graph communities (right)
net7 = Network(height="700px", width="100%", directed=False, bgcolor="#1a1a2e", font_color="white")
net7.barnes_hut(gravity=-5000, central_gravity=0.3, spring_length=150)

# Color by status
task_status_colors = {"open": "#F5A623", "in_progress": "#4A90D9", "queued": "#7ED321", "partial": "#BD10E0"}

# Add task nodes (left side)
for task in v["tasks"][:30]:  # top 30 for readability
    color = task_status_colors.get(task["status"], "#CCCCCC")
    label = f"{task['ref'][:20]}\n({task['status']})"
    net7.add_node(task["ref"], label=label, color=color, x=-200, size=25, font={"size": 10},
                  title=f"Status: {task['status']}\nConnectivity: {task['connectivity']}\nMatched: {task['matched']}")

# Add community nodes (right side) — top 5 communities
comm = communities(g, ("contradicts", "supersedes"))
comm_sizes = Counter(comm.values())
top5_comms = [cid for cid, _ in comm_sizes.most_common(5)]
comm_colors = ["#FF6B6B", "#4A90D9", "#7ED321", "#F5A623", "#BD10E0"]

for i, cid in enumerate(top5_comms):
    net7.add_node(f"COMM_{cid}", label=f"Community {cid}\n({comm_sizes[cid]} nodes)",
                  color=comm_colors[i], x=200, y=i*100-200, size=40, font={"size": 12, "bold": True})

# Connect tasks to communities via sample_nodes
for task in v["tasks"][:30]:
    for sample_node in task.get("sample_nodes", [])[:3]:
        node_comm = comm.get(sample_node)
        if node_comm in top5_comms:
            net7.add_edge(task["ref"], f"COMM_{node_comm}", color="#888888", width=1)

net7.save_graph("/tmp/nb13_v7_bridge_bipartite.html")
net7.show("/tmp/nb13_v7_bridge_bipartite.html", notebook=False)
print(f"V7: Bridge bipartite — /tmp/nb13_v7_bridge_bipartite.html")
print(f"  {min(30, len(v['tasks']))} tasks ↔ {len(top5_comms)} communities")

In [ ]:
# CELL 8: V8 — Capas grafo completo multi-layer
# Show all edge kinds as separate layers
edge_kinds = Counter(e["kind"] for e in g["edges"])
print(f"Edge kinds: {dict(edge_kinds)}")

# Create a multi-panel figure with subgraphs per layer
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("Grafo Epistémico — Capas por Edge Kind", fontsize=14, fontweight="bold")

layer_kinds = ["child_of", "extracted_from", "certified_by", "answers", "contradicts", "supersedes", "correlates_with", "derived_from"]
layer_colors = {"child_of": "#8B572A", "extracted_from": "#50E3C2", "certified_by": "#7ED321",
                "answers": "#4A90D9", "contradicts": "#FF6B6B", "supersedes": "#F5A623",
                "correlates_with": "#BD10E0", "derived_from": "#9013FE"}

for i, (kind, ax) in enumerate(zip(layer_kinds, axes.flat)):
    edges_of_kind = [e for e in g["edges"] if e["kind"] == kind][:100]  # limit
    nodes_in_layer = set()
    for e in edges_of_kind:
        nodes_in_layer.add(e["source"])
        nodes_in_layer.add(e["target"])
    
    if not nodes_in_layer:
        ax.text(0.5, 0.5, f"{kind}\n(0 edges)", ha="center", va="center", fontsize=10)
        ax.set_title(f"{kind} ({edge_kinds.get(kind, 0)})")
        ax.axis("off")
        continue
    
    H = nx.DiGraph()
    for n_id in list(nodes_in_layer)[:50]:
        node = next((n for n in g["nodes"] if n["id"] == n_id), None)
        kind_node = node.get("kind", "?") if node else "?"
        H.add_node(n_id, kind=kind_node)
    for e in edges_of_kind[:50]:
        if e["source"] in H and e["target"] in H:
            H.add_edge(e["source"], e["target"])
    
    pos = nx.spring_layout(H, k=1/np.sqrt(len(H.nodes())), seed=42, iterations=30)
    color = layer_colors.get(kind, "#CCCCCC")
    nx.draw_networkx_nodes(H, pos, node_color=color, node_size=100, ax=ax, alpha=0.8)
    nx.draw_networkx_edges(H, pos, edge_color="#888888", arrows=True, arrowsize=8, ax=ax, alpha=0.5)
    ax.set_title(f"{kind} ({edge_kinds.get(kind, 0)} total)", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("/tmp/nb13_v8_layers.png", dpi=100, bbox_inches="tight")
plt.show()
print(f"V8: 8 layers — /tmp/nb13_v8_layers.png")
print(f"  Edge kinds: {dict(edge_kinds)}")

In [ ]:
# CELL 9: SUMMARY — qué explorar más a fondo
print("=" * 80)
print("NB13 RESUMEN — Visualizaciones generadas")
print("=" * 80)
print()
print("V1: Fierro argumentation tree (Q→Hip→Evid→Resp)")
print("    → /tmp/nb13_v1_fierro_tree.html")
print("    → 3 objections, all RESUELTA")
print()
print("V2: Predicate trees (OR/AND parsed)")
print("    → /tmp/nb13_v2_predicate_trees.html")
print(f"    → {len(or_preds)} predicates visualized")
print()
print("V3: Certification chain (doc→claim→cert→status)")
print("    → /tmp/nb13_v3_cert_chain.html")
print(f"    → {len(cert_sub)} nodes, {len(cert_edges)} edges")
print()
print("V4: Supersedes timeline (claim evolution)")
print("    → /tmp/nb13_v4_supersedes_timeline.html")
print(f"    → {len(sup_nodes_ids)} nodes, {len(supersedes_edges)} edges, depth 0-{max_depth}")
print()
print("V5: Communities (71 clusters)")
print("    → /tmp/nb13_v5_communities.html")
print(f"    → Top 15 communities shown")
print()
print("V6: 3 Heatmaps (source × kind/status/risk)")
print("    → /tmp/nb13_v6_heatmaps.png")
print(f"    → {len(top_sources)} sources")
print()
print("V7: Bridge bipartite (tasks ↔ communities)")
print("    → /tmp/nb13_v7_bridge_bipartite.html")
print(f"    → {min(30, len(v['tasks']))} tasks ↔ {len(top5_comms)} communities")
print()
print("V8: 8 layers (all edge kinds)")
print("    → /tmp/nb13_v8_layers.png")
print(f"    → {dict(edge_kinds)}")
print()
print("=" * 80)
print("PRÓXIMO PASO: Explorar HTMLs interactivos en browser")
print("  Abrir cada /tmp/nb13_v*.html para zoom/pan/hover/click")
print("  Seleccionar las visualizaciones más significativas")
print("  Profundizar con análisis detallado")
print("=" * 80)